# Prompt 5: Broadcasting & Optimization
## Databricks Certified Associate Developer for Apache Spark
### Topic 1 — Apache Spark Architecture & Components (20%)

---

Broadcasting eliminates shuffle-heavy **SortMergeJoins** by sending a *small* dataset to every executor exactly once,
allowing each executor to perform the join locally — with no network shuffle of the large table.

**Exam facts to memorise:**
- Default auto-broadcast threshold: **10 MB** (`spark.sql.autoBroadcastJoinThreshold`)
- Broadcast join algorithm: **BroadcastHashJoin** (vs SortMergeJoin for large-large joins)
- Force broadcast with `broadcast()` hint from `pyspark.sql.functions`
- Disable auto-broadcast: set threshold to `-1`
- Two broadcast types: **broadcast joins** (DataFrames) and **broadcast variables** (arbitrary Python objects)
- Broadcast variables accessed inside UDFs with `.value`
- Cleanup with `bc.unpersist()` or `bc.destroy()`

## 1. What is Broadcasting?

In a standard Spark join between a large table and a small table, Spark may:
1. Hash-partition **both** tables by the join key (shuffle both sides)
2. Send matching partitions to the same executor
3. Perform a **SortMergeJoin** — expensive because the large side is fully shuffled

With broadcasting, Spark instead:
1. Collects the **small** table to the Driver
2. Serialises it into a broadcast object
3. Ships it to **every** executor once (BitTorrent-style peer distribution)
4. Each executor performs a **local hash lookup** against its slice of the large table — no shuffle of the large table at all

```
WITHOUT Broadcast:                    WITH Broadcast:
+-----------------+                   +-----------------+
| Large Table     |--shuffle---+       | Large Table     |--NO shuffle--+
+-----------------+            |       +-----------------+              |
                  SortMergeJoin|                         BroadcastHash  |
+-----------------+            |       +-----------------+  broadcast   |Join
| Small Table     |--shuffle---+       | Small Table     |-----------> each
+-----------------+                   +-----------------+ (all execs) executor
```

### When to use broadcasting

| Condition | Use Broadcast? |
|-----------|----------------|
| Small table <= 10 MB (default threshold) | Yes — auto-broadcast happens automatically |
| Small table > 10 MB but fits in executor memory | Yes — use `broadcast()` hint explicitly |
| Both tables are GB-scale | No — SortMergeJoin or enable AQE |
| Lookup dict / config reused in UDFs | Yes — `sc.broadcast(dict)` |
| Table changes frequently at runtime | No — broadcast is immutable |
| Full outer join | No — BroadcastHashJoin not supported for FULL OUTER |

> **Rule of thumb:** If one side of your join comfortably fits in executor JVM heap (commonly up to a few hundred MB with tuning), broadcasting it beats any shuffle-based join.

## 2. Two Types of Broadcasting in PySpark

### Type A: Broadcast Joins (DataFrame API / Spark SQL)
Used when joining two DataFrames where one is small.
- `broadcast()` hint from `pyspark.sql.functions`
- Forces **BroadcastHashJoin** regardless of table size
- Visible in `explain()` output as `BroadcastHashJoin`

```python
from pyspark.sql.functions import broadcast
result = large_df.join(broadcast(small_df), on='key')
```

Also available as a SQL hint:
```sql
SELECT /*+ BROADCAST(small_table) */ * FROM large_table JOIN small_table USING (key)
```

### Type B: Broadcast Variables (SparkContext)
Used to distribute a read-only Python object (dict, list, set, config) to all executors efficiently.
- `spark.sparkContext.broadcast(value)` — creates a `Broadcast[T]` wrapper
- Access the wrapped value inside UDFs with `.value`
- Sent once; cached on each executor for the life of the application
- Far more efficient than closing over a large dict in a UDF (which re-serialises it per task)

```python
lookup = {'NY': 'New York', 'CA': 'California', 'TX': 'Texas'}
bc_lookup = spark.sparkContext.broadcast(lookup)
# Inside UDF: bc_lookup.value.get(abbrev)
```

### Key differences

| | Broadcast Join | Broadcast Variable |
|---|---|---|
| **What is broadcast** | A whole small DataFrame | Any Python object (dict, list, set, etc.) |
| **API** | `broadcast(df)` hint | `sc.broadcast(obj)` |
| **Spark executes as** | BroadcastHashJoin in physical plan | Executor-side `.value` lookup in UDF |
| **Best for** | DataFrame joins | UDF lookups, config distribution |
| **Memory release** | Automatic after job | `bc.unpersist()` or `bc.destroy()` |

In [ ]:
# Cell 1: Setup — SparkSession and sample DataFrames
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, col, udf
from pyspark.sql.types import StringType, BooleanType

spark = SparkSession.builder \
    .appName('BroadcastOptimization') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Large table: 10,000 sales transactions (represents millions in production)
large_data = [(i, f'PROD-{i % 100:03d}', float(i * 1.5)) for i in range(1, 10001)]
large_df = spark.createDataFrame(large_data, ['txn_id', 'product_code', 'amount'])

# Small table: 100-row product catalogue — perfect candidate for broadcast
categories = ['Electronics', 'Clothing', 'Food']
small_data = [(f'PROD-{i:03d}', f'Product {i}', categories[i % 3]) for i in range(100)]
small_df = spark.createDataFrame(small_data, ['product_code', 'product_name', 'category'])

print(f'Large table row count: {large_df.count():,}')
print(f'Small table row count: {small_df.count():,}')
large_df.printSchema()
small_df.printSchema()

In [ ]:
# Cell 2: Regular join WITHOUT broadcast — observe SortMergeJoin in physical plan

# Disable auto-broadcast so Spark is forced to use SortMergeJoin
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

regular_join = large_df.join(small_df, on='product_code', how='inner')

print('=== EXPLAIN: Regular Join (auto-broadcast DISABLED) ===')
print('Look for: SortMergeJoin in the physical plan')
print()
regular_join.explain()  # Physical plan — will show SortMergeJoin

# Expected output includes a line like:
# +- SortMergeJoin [product_code#...], [product_code#...], Inner

# Re-enable auto-broadcast at 10 MB default
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))
print('Auto-broadcast threshold reset to 10 MB')

In [ ]:
# Cell 3: Broadcast join WITH broadcast() hint — observe BroadcastHashJoin

# Disable auto-broadcast again to ensure the hint (not auto-threshold) triggers broadcast
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

broadcast_join = large_df.join(broadcast(small_df), on='product_code', how='inner')

print('=== EXPLAIN: Broadcast Join (broadcast() hint applied) ===')
print('Look for: BroadcastHashJoin in the physical plan')
print()
broadcast_join.explain()  # Physical plan — will show BroadcastHashJoin

# Expected output includes a line like:
# +- BroadcastHashJoin [product_code#...], [product_code#...], Inner, BuildRight

# Both joins produce identical results — the difference is only in execution strategy
print()
print('Sample results (same data as regular join):')
broadcast_join.select('txn_id', 'product_name', 'category', 'amount').show(5)

# Re-enable default
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))

In [ ]:
# Cell 4: Broadcast Variable — distribute a lookup dictionary to all executors
# Use case: expand state abbreviations inside a UDF

state_lookup = {
    'NY': 'New York',     'CA': 'California',  'TX': 'Texas',
    'FL': 'Florida',      'WA': 'Washington',  'IL': 'Illinois',
    'PA': 'Pennsylvania', 'OH': 'Ohio',         'GA': 'Georgia'
}

# Broadcast the dict to all executors — sent ONCE, cached per executor JVM
bc_state = spark.sparkContext.broadcast(state_lookup)

# UDF references the broadcast variable (NOT the dict directly)
# Closing over the dict directly re-serialises it with every task — very slow
@udf(StringType())
def expand_state(abbrev):
    if abbrev is None:
        return None
    return bc_state.value.get(abbrev, 'Unknown')  # .value accesses the wrapped dict

customers = spark.createDataFrame(
    [('Alice', 'NY'), ('Bob', 'CA'), ('Charlie', 'TX'), ('Diana', 'ZZ'), ('Eve', None)],
    ['name', 'state_abbrev']
)

result = customers.withColumn('state_full', expand_state(col('state_abbrev')))
result.show()

# Cleanup: release executor memory when broadcast variable is no longer needed
bc_state.unpersist()
print('Broadcast variable unpersisted.')

In [ ]:
# Cell 5: Auto-broadcast threshold configuration
# Spark auto-broadcasts any table whose estimated size <= this threshold

# --- Check current threshold ---
current = spark.conf.get('spark.sql.autoBroadcastJoinThreshold')
print(f'Default autoBroadcastJoinThreshold: {current} bytes ({int(current)/1024/1024:.1f} MB)')

# --- Raise threshold to 50 MB (session-level) ---
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(50 * 1024 * 1024))
print('Raised threshold to 50 MB — any table <= 50 MB auto-broadcasts')

# --- Disable auto-broadcast entirely ---
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
print('Auto-broadcast disabled (-1) — must use broadcast() hint manually')

# --- Set in spark-submit (application-level) ---
# spark-submit --conf spark.sql.autoBroadcastJoinThreshold=52428800 my_app.py

# --- Set in SparkSession builder ---
# spark = SparkSession.builder \
#     .config('spark.sql.autoBroadcastJoinThreshold', 50 * 1024 * 1024) \
#     .getOrCreate()

# EXAM NOTE: Spark's size estimate is based on internal statistics, not raw file size.
# To check estimated size: run df.explain(True) and look for 'Statistics(sizeInBytes=...)'

# Reset to default for remainder of notebook
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))
print('Reset to default 10 MB')

In [ ]:
# Cell 6: AQE (Adaptive Query Execution) and runtime broadcasting
# AQE can promote a SortMergeJoin to BroadcastHashJoin at RUNTIME
# after shuffle map stages complete and actual partition sizes are known

# Enable AQE (default in Spark 3.x)
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.autoBroadcastJoinThreshold', str(30 * 1024 * 1024))  # 30 MB

print('AQE enabled:', spark.conf.get('spark.sql.adaptive.enabled'))
print('AQE broadcast threshold:', spark.conf.get('spark.sql.adaptive.autoBroadcastJoinThreshold'))

# Scenario: a table that is large BEFORE filtering becomes small AFTER
# Without AQE: Spark plans SortMergeJoin based on pre-filter size estimates
# With AQE:    Spark re-evaluates after the filter stage completes and may switch to BroadcastHashJoin
filtered_small = large_df.filter(col('txn_id') < 10)  # Only 9 rows after filter
probe_table = spark.range(100).withColumnRenamed('id', 'txn_id')

aqe_join = probe_table.join(filtered_small, on='txn_id')
print()
print('=== EXPLAIN: AQE join (may auto-upgrade to BroadcastHashJoin at runtime) ===')
aqe_join.explain(mode='formatted')

# EXAM POINT: AQE's adaptive broadcast does NOT require a broadcast() hint.
# It fires automatically based on runtime statistics collected after each shuffle stage.
# Key configs:
#   spark.sql.adaptive.enabled                   = true   (enable AQE)
#   spark.sql.adaptive.autoBroadcastJoinThreshold = 30MB   (AQE broadcast size limit)

## 3. Performance Benefits

### BroadcastHashJoin vs SortMergeJoin

| Aspect | SortMergeJoin | BroadcastHashJoin |
|--------|--------------|--------------------|
| **Both sides shuffle?** | Yes — both shuffled by join key | No — only small side transferred once |
| **Sort required?** | Yes — both sides sorted | No — hash lookup (O(1)) |
| **Extra stage?** | Yes — shuffle boundary = new stage | No extra stage for join |
| **Shuffle write bytes** | Full size of both tables | Zero for large table |
| **Memory pattern** | Shuffle spill to disk possible | Small table in-memory on every executor |
| **Best for** | Large-large joins (no alternative) | Large-small joins |
| **Supported join types** | All (inner, left, right, full outer) | Inner, left, left_semi, left_anti only |

### Typical speedup
Replacing SortMergeJoin with BroadcastHashJoin on a **100 GB x 10 MB** join:
- Stages: 3 → 2 (shuffle map stage for join eliminated)
- Shuffle write: 100+ GB → 0 bytes for the join
- Wall-clock time: **5–20x speedup** depending on cluster network and executor count

### Broadcast Variable vs Closing Over a Dict in a UDF

| Approach | Serialisation overhead |
|----------|------------------------|
| `sc.broadcast(lookup_dict)` | Serialised **once per executor** — cached for all tasks on that executor |
| Closing over `lookup_dict` directly | Serialised with **every single task** — potentially millions of serialisations |

With 10 executors and 1,000 tasks per executor, broadcast saves 9,990 serialisations.

## 4. Best Practices and Limitations

### Best Practices

1. **Filter before broadcasting** — apply all possible filters to the small table *before* broadcasting; every byte saved reduces network transfer and executor memory consumption
2. **Verify with explain()** — after adding a `broadcast()` hint, run `df.explain()` and confirm `BroadcastHashJoin` appears in the physical plan; if not, diagnose why
3. **Enable AQE** — `spark.sql.adaptive.enabled=true` (default in Spark 3.x) gives runtime broadcast promotion without needing explicit hints
4. **Raise threshold for known-small tables** — if a dimension table is consistently 20–50 MB, raise `autoBroadcastJoinThreshold` rather than adding hints to every query
5. **Broadcast variables over direct dict closure** — always use `sc.broadcast()` for any large lookup object in a UDF; never close over a large collection directly
6. **Unpersist when done** — call `bc.unpersist()` (keeps on Driver for rebroadcast) or `bc.destroy()` (removes everywhere) to free executor heap memory

### Limitations

| Limitation | Detail |
|------------|--------|
| **Driver OOM** | Driver must collect the entire small table into JVM heap before broadcasting; if table is actually large, Driver will OOM |
| **Executor memory** | All executors hold broadcast table simultaneously; large broadcast x many executors = high aggregate memory |
| **Immutable** | Broadcast variables are read-only; cannot be updated or modified after creation |
| **No streaming-to-streaming** | In Structured Streaming, one side of a broadcast join must be a static (non-streaming) DataFrame |
| **Full outer join** | `BroadcastHashJoin` does not support FULL OUTER joins; Spark silently falls back to SortMergeJoin |
| **Hint may be ignored** | If Spark determines broadcasting would exceed available memory, it may fall back to SortMergeJoin |

### Quick Reference: Broadcast Decision Tree

```
Is the join type FULL OUTER?           YES -> Cannot broadcast. Use SortMergeJoin.
                                        NO
                                         |
Is the small table < 10 MB?            YES -> Auto-broadcast (no hint needed)
                                        NO
                                         |
Does it fit in executor memory?         YES -> Use broadcast() hint explicitly
                                        NO
                                         |
Can you filter it down first?           YES -> Filter then broadcast()
                                        NO
                                         |
Enable AQE and let Spark decide.       (spark.sql.adaptive.enabled=true)
```

## 5. Real-World Scenarios

<details>
<summary>Scenario 1: E-Commerce — Enriching 500M Transactions with a 5k Product Catalogue</summary>

**Situation:**
A retailer has a `transactions` table with 500 million rows (200 GB) and a `products` catalogue with 5,000 rows (~500 KB).
A daily ETL job joins them by `product_id` to add `product_name` and `category` to each transaction.
Without broadcasting, Spark plans a SortMergeJoin — shuffling 200 GB across the network.

**PySpark Code:**
```python
from pyspark.sql.functions import broadcast

transactions = spark.read.parquet('/data/transactions/')   # 200 GB
products     = spark.read.parquet('/data/products/')       # ~500 KB — tiny

# broadcast() hint forces BroadcastHashJoin — 200 GB shuffle eliminated
enriched = transactions.join(broadcast(products), on='product_id', how='left')
enriched.write.parquet('/data/enriched_transactions/', mode='overwrite')

# Verify the plan
transactions.join(broadcast(products), on='product_id').explain()
# Expected: +- BroadcastHashJoin [product_id#...], [product_id#...], LeftOuter, BuildRight
```

**Expected Outcome:**
Physical plan shows `BroadcastHashJoin`. No shuffle stage for the join.
ETL time drops from ~45 minutes (SortMergeJoin) to ~8 minutes (BroadcastHashJoin).
Shuffle write bytes: 200+ GB → 0 bytes for the join operation.

**Exam Sub-topic:** `broadcast()` hint; BroadcastHashJoin vs SortMergeJoin; `autoBroadcastJoinThreshold`
</details>

<details>
<summary>Scenario 2: Fraud Detection — Broadcasting a Set of Blocked Account IDs in a UDF</summary>

**Situation:**
A fraud detection pipeline processes millions of payment events per batch.
At startup, 50,000 blocked account IDs are loaded from a database and used inside a UDF to flag fraudulent transactions.
Without `sc.broadcast()`, each of the millions of tasks re-serialises the 50k-entry set — wasting CPU and memory.

**PySpark Code:**
```python
from pyspark.sql.functions import udf, col
from pyspark.sql.types import BooleanType

# Load blocked IDs at Driver (single collect to Python set)
blocked_ids = {row['account_id'] for row in spark.read.table('blocked_accounts').collect()}

# Broadcast to all executors — serialised ONCE per executor, not per task
bc_blocked = spark.sparkContext.broadcast(blocked_ids)

@udf(BooleanType())
def is_blocked(account_id):
    return account_id in bc_blocked.value  # O(1) set lookup using .value

payments = spark.read.table('payments')
flagged  = payments.withColumn('is_fraud', is_blocked(col('account_id')))
flagged.filter(col('is_fraud')).write.table('fraud_alerts')

bc_blocked.unpersist()  # Release executor memory when no longer needed
```

**Expected Outcome:**
The 50k-set is serialised once per executor (10 executors = 10 serialisations).
Without broadcast, 1,000 tasks x 10 executors = 10,000 serialisations.
Throughput improves significantly; GC pressure from repeated large object allocation is removed.

**Exam Sub-topic:** Broadcast variables; `sc.broadcast()`; `.value` accessor; `unpersist()`
</details>

<details>
<summary>Scenario 3: Reporting — Raising the Threshold for a 40 MB Dimension Table</summary>

**Situation:**
A data warehouse team runs dozens of daily reports joining a 1 TB `orders` fact table with a `customers`
dimension table of 40 MB. The default 10 MB threshold means Spark uses SortMergeJoin for all of them.
The team wants auto-broadcast without adding `broadcast()` hints to every query.

**Approaches:**
```python
# Option A: Session-level (affects all joins in this Spark session)
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(50 * 1024 * 1024))  # 50 MB

# Option B: Application-level via spark-submit
# spark-submit --conf spark.sql.autoBroadcastJoinThreshold=52428800 report_job.py

# Option C: SparkSession builder (baked into the application)
spark = SparkSession.builder \
    .config('spark.sql.autoBroadcastJoinThreshold', 50 * 1024 * 1024) \
    .getOrCreate()

# Now Spark auto-broadcasts customers (40 MB < 50 MB threshold) — no hints needed
orders    = spark.read.parquet('/data/orders/')     # 1 TB
customers = spark.read.parquet('/data/customers/')  # 40 MB
report    = orders.join(customers, on='customer_id')
report.explain()  # Confirm BroadcastHashJoin appears
```

**Expected Outcome:**
All joins where one side is <= 50 MB are automatically broadcast — no per-query changes.
Daily report runtime across the team drops from ~30 min to ~5 min.

**Exam Sub-topic:** `spark.sql.autoBroadcastJoinThreshold`; session vs application-level config; auto-broadcast
</details>

<details>
<summary>Scenario 4: AQE — Dynamically Upgrading to BroadcastHashJoin After Runtime Filter</summary>

**Situation:**
A pipeline joins `web_events` (500 GB) with `active_campaigns` (2 GB total, but after filtering for
`campaign_type = 'EMAIL'` only 8 MB remain). Without AQE, Spark plans SortMergeJoin based on the
pre-filter 2 GB size estimate. With AQE, Spark detects the small post-filter size at runtime
and promotes to BroadcastHashJoin — automatically, without any hint.

**PySpark Code:**
```python
# AQE is default-enabled in Spark 3.x
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.autoBroadcastJoinThreshold', str(20 * 1024 * 1024))  # 20 MB

web_events      = spark.read.parquet('/data/web_events/')    # 500 GB
campaigns       = spark.read.parquet('/data/campaigns/')     # 2 GB total
email_campaigns = campaigns.filter(col('campaign_type') == 'EMAIL')  # ~8 MB after filter

# NO broadcast() hint — AQE decides at runtime
result = web_events.join(email_campaigns, on='campaign_id')
result.write.parquet('/output/email_events/')
```

**Expected Outcome:**
AQE's `OptimizeShuffleWithLocalRead` and `DynamicJoinSelection` rules detect that `email_campaigns` is
8 MB after the filter stage, re-plan as BroadcastHashJoin, and eliminate the `web_events` shuffle entirely.
No code change required — AQE handles it automatically.

**Exam Sub-topic:** AQE; `spark.sql.adaptive.enabled`; `spark.sql.adaptive.autoBroadcastJoinThreshold`; runtime join promotion
</details>

<details>
<summary>Scenario 5: Pitfall — Driver OOM from Over-Broadcasting a Large Table</summary>

**Situation:**
A developer notices a slow join and adds a `broadcast()` hint. They assume the 'small' side is small,
but it is actually 8 GB. The Driver attempts to collect 8 GB into its JVM heap and crashes with an OOM error.

**Wrong Code (causes Driver OOM):**
```python
# WRONG: large_dim is 8 GB — way too large to broadcast
large_dim = spark.read.parquet('/data/large_dimension/')  # 8 GB
fact       = spark.read.parquet('/data/fact/')

# Driver will attempt to collect 8 GB into heap — OOM crash
result = fact.join(broadcast(large_dim), on='dim_key')
result.show()  # java.lang.OutOfMemoryError: GC overhead limit exceeded
```

**Correct Approaches:**
```python
# Option 1: Remove hint — let Spark use SortMergeJoin (safe for large-large joins)
result = fact.join(large_dim, on='dim_key')

# Option 2: Filter large_dim BEFORE joining — reduce size enough to broadcast safely
filtered_dim = large_dim.filter(col('active') == True)  # 8 GB -> ~50 MB
result = fact.join(broadcast(filtered_dim), on='dim_key')  # Now safe

# Option 3: Let AQE decide — no hint, Spark chooses at runtime
spark.conf.set('spark.sql.adaptive.enabled', 'true')
result = fact.join(large_dim, on='dim_key')  # AQE picks the best join strategy
```

**Diagnosis — check estimated size before broadcasting:**
```python
# Run explain(True) and look for 'Statistics(sizeInBytes=...)'
large_dim.explain(True)
# Statistics(sizeInBytes=8.0 GiB, ...)  <-- do NOT broadcast this
```

**Expected Outcome:**
Removing the inappropriate `broadcast()` hint eliminates the OOM.
Lesson: broadcasting is powerful but must only be used when the small table truly fits in both
Driver heap (for collection) and all executor heaps (for caching).

**Exam Sub-topic:** Broadcast limitations; Driver OOM; filter-before-broadcast; diagnosing with `explain(True)`
</details>

## 6. Exam Cheat Sheet

### Key Facts

| Fact | Value |
|------|-------|
| Default broadcast threshold | **10 MB** |
| Config key | `spark.sql.autoBroadcastJoinThreshold` |
| Disable auto-broadcast | Set to `-1` |
| Force broadcast in DataFrame API | `large_df.join(broadcast(small_df), on='key')` |
| Force broadcast in SQL | `SELECT /*+ BROADCAST(t) */ ...` |
| Join algorithm when broadcast used | `BroadcastHashJoin` |
| Join algorithm without broadcast (large-large) | `SortMergeJoin` |
| Broadcast variable creation | `bc = spark.sparkContext.broadcast(obj)` |
| Broadcast variable access inside UDF | `bc.value` |
| Memory release | `bc.unpersist()` (keeps on Driver) or `bc.destroy()` (removes everywhere) |
| AQE enables | `spark.sql.adaptive.enabled=true` |
| AQE broadcast threshold | `spark.sql.adaptive.autoBroadcastJoinThreshold` |
| Full outer join + broadcast | Not supported — hint ignored, falls back to SortMergeJoin |
| Supported join types for BroadcastHashJoin | inner, left, left_semi, left_anti |

---

## 7. Practice Q&A

<details>
<summary>Q1: What is the default value of spark.sql.autoBroadcastJoinThreshold and what does it control?</summary>

**A:** The default is **10 MB** (10 * 1024 * 1024 = 10,485,760 bytes). It controls the maximum estimated
DataFrame size that Spark will automatically broadcast when planning a join. If one side of a join has an
estimated size at or below this threshold, Spark uses BroadcastHashJoin automatically — no `broadcast()`
hint needed. Set to `-1` to disable automatic broadcasting entirely.
</details>

<details>
<summary>Q2: What join algorithm does Spark use when broadcast() hint is applied, and how does it differ from SortMergeJoin?</summary>

**A:** Spark uses **BroadcastHashJoin**.

BroadcastHashJoin:
1. Collects the small table to the Driver
2. Serialises it and distributes it to every executor via BitTorrent-style broadcast
3. Each executor performs a hash lookup against the large table partitions it already holds — no shuffle

SortMergeJoin:
1. Shuffles both tables by the join key (both cross the network)
2. Sorts both sides by the join key
3. Performs a merge scan

Key difference: BroadcastHashJoin eliminates all shuffle and sort for the large table, removing an
entire stage boundary and potentially gigabytes of network I/O.
</details>

<details>
<summary>Q3: What is the difference between a broadcast join (broadcast() hint) and a broadcast variable (sc.broadcast())?</summary>

**A:**
- **Broadcast join** — `broadcast(df)` hint distributes an entire *DataFrame* to all executors for use
  in a join operation. Spark internally uses BroadcastHashJoin. Best for joining a large table with a
  small dimension table.
- **Broadcast variable** — `sc.broadcast(obj)` distributes any arbitrary Python object (dict, list, set,
  config) to all executors. Accessed inside UDFs via `.value`. Best for lookup tables, constant data, or
  config objects used in UDF logic.

They solve different problems: broadcast join eliminates shuffle in join operations;
broadcast variables eliminate repeated serialisation of large objects per task in UDFs.
</details>

<details>
<summary>Q4: A developer applies broadcast() hint but the physical plan still shows SortMergeJoin. What are the possible causes?</summary>

**A:** Possible causes:
1. **Table is too large for available memory** — Spark determines that broadcasting would risk OOM and
   falls back to SortMergeJoin to avoid crashing executors or the Driver
2. **Full outer join** — BroadcastHashJoin does not support FULL OUTER joins; Spark ignores the hint
   silently and uses SortMergeJoin
3. **Statistics not available** — Without ANALYZE TABLE or Adaptive Query Execution, Spark may not know
   the table is small and ignores the hint
4. **AQE re-planning** — In rare cases AQE may override the plan based on runtime conditions

Diagnosis: run `df.explain(True)` and check `Statistics(sizeInBytes=...)` for the small table.
</details>

<details>
<summary>Q5: How does Adaptive Query Execution (AQE) relate to broadcasting? When does it help most?</summary>

**A:** AQE can dynamically **upgrade** a planned SortMergeJoin to a BroadcastHashJoin at *runtime*,
after shuffle map stages complete and Spark has accurate size statistics from the actual data.

AQE helps most when:
- A table is large in the plan (Spark estimated SortMergeJoin) but small after a filter runs
- Pre-execution size statistics were inaccurate or unavailable
- You want automatic optimisation without adding `broadcast()` hints to every query

Key configs:
- `spark.sql.adaptive.enabled = true` (default in Spark 3.x)
- `spark.sql.adaptive.autoBroadcastJoinThreshold` — AQE broadcast threshold (separate from static threshold)

AQE is complementary to (not a replacement for) explicit `broadcast()` hints.
Hints fire at plan time; AQE fires at runtime after real statistics are available.
</details>